In [46]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings


Loading Document

In [47]:
with open("AI_psychology_medium.txt", "r", encoding="utf-8") as f:
    text = f.read()

Splitting the Document

In [48]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 4


Create embeddings

In [49]:
from langchain_groq import ChatGroq

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 110.55it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Store chunks in FAISS vector database

In [50]:
vectorstore = FAISS.from_texts(
    texts=chunks,
    embedding=embeddings
)

Create retriever

In [51]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

Summarization prompt

In [81]:
from langchain_core.prompts import PromptTemplate

summary_prompt = PromptTemplate(
    input_variables=["context"],
    template="""
You are a professional summarizer.

STRICT RULES:
- Output MUST be at most 4 bullet points
- Each bullet point ≤ 12 words
- Do NOT add explanations
- Do NOT repeat information
- Give it in crisp manner 
- I dont want lenghty answer short it  

Context:
{context}

Very Short Summary:
"""
)

# from langchain_core.prompts import PromptTemplate

# summary_prompt = PromptTemplate(
#     input_variables=["context"],
#     template="""
# TASK: Produce a VERY SHORT summary.

# STRICT RULES (MANDATORY):
# - Output EXACTLY ONE paragraph
# - Maximum 40 words
# - bullet points
# - No headings
# - No numbering
# - No extra explanations

# Context:
# {context}

# Short Summary:
# """
# )

Initialize LLM

In [82]:
llm =  ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=60,
    api_key=os.getenv("GROQ_API_KEY")
)

Build LCEL RAG summarization chain

In [83]:
rag_chain = (
    {"context": retriever}
    | summary_prompt
    | llm
    | StrOutputParser()
)

Run summarization

In [84]:
summary = rag_chain.invoke("Summarize the document")

print(summary)

• Humans easily blur line between human and machine intelligence.
• AI learns from human data, influencing future behavior.
• AI will augment human intelligence, not replace it.
• Responsible AI development requires fairness and transparency.


In [87]:
import json
from datetime import datetime

HISTORY_FILE = "history.json"

def save_history(query, summary):
    record = {
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "summary": summary
    }

    try:
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            history = json.load(f)
            if not isinstance(history, list):
                history = []
    except (FileNotFoundError, json.JSONDecodeError):
        history = []

    history.append(record)

    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4, ensure_ascii=False)

In [88]:
query = "Summarize the document"
summary = rag_chain.invoke(query)

save_history(query, summary)

print(summary)

• Humans easily blur line between human and machine intelligence.
• AI creates feedback loop with humans, influencing behavior.
• AI will augment human intelligence, not replace it.
• Understanding human psychology is key to AI design.


In [62]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

print(llm.invoke("Say hello in one sentence"))

content='Hello, how are you today?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 40, 'total_tokens': 48, 'completion_time': 0.016084303, 'completion_tokens_details': None, 'prompt_time': 0.00274722, 'prompt_tokens_details': None, 'queue_time': 0.046264787, 'total_time': 0.018831523}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019c6f3a-26a5-7312-afde-11475f48e410-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 8, 'total_tokens': 48}
